# Custom CNN Evaluation

In [ ]:
# ==============================================================================
# THESIS-READY MULTI-SEED FEED CLASSIFICATION PIPELINE
# DIRECTML-SAFE EXTENSION FOR WINDOWS + AMD GPU
# REGIME: TRUE LINEAR PROBE WITH FROZEN BACKBONES & CACHED EMBEDDINGS
# COMPARISON: UNPERTURBED LEJEPA PRE-TRAINED BACKBONE vs. PURE RANDOM INITIALIZATION
# FOCUS: 6-CLASS FORAGE CLASSIFICATION ONLY (1-SHOT, 10-SHOT, FULL)
# ==============================================================================

import os
import copy
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

from torchvision.transforms import v2
from PIL import Image, ImageFile

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Required for 3D projection plots

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True


# ==============================================================================
# =============================== CONFIGURATION ================================
# ==============================================================================

CLASSIFICATION_DATASET_ROOT = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\Feed Classification dataset_\Classification Dataset\Classification dataset copy 3"
JEPA_CKPT = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\iMAGENET+FORAGENET PRETRAINING\lejepa_convnet_encoder_tile_loss.pth"
OUTPUT_ROOT = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\Figure"

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 60

LR = 0.001788039218013555
WEIGHT_DECAY = 0.005291842753172914
BASE_SEED = 42
MULTIPLE_SEEDS = [0, 1, 2, 3, 4]  # 5 validation seeds for stabilized few-shot metrics
NUM_WORKERS = 0
VAL_FRAC = 0.2

CLASSIFICATION_REGIMES = {
    "one_shot": 1,
    "ten_shot": 10,
    "full": None,
}

# The two backbone initialization modes to evaluate and export sequentially
INIT_MODES_TO_RUN = ["jepa", "random"]

ZOOM_CLASSES = ["alfalfa", "haylage", "tmr"]
CHOSEN_ZOOM = 0.15
GRID_IMAGE_SIZE = (400, 400)
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")


# ==============================================================================
# ============================== DEVICE ROUTINES ===============================
# ==============================================================================

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda"), "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps"), "mps"
    try:
        import torch_directml
        return torch_directml.device(), "directml"
    except Exception:
        pass
    return torch.device("cpu"), "cpu"

DEVICE, DEVICE_NAME = get_device()

def move_to_device(x):
    return x.to(DEVICE)


# ==============================================================================
# ============================= UTILITY METHODS ================================
# ==============================================================================

def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(BASE_SEED)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def show_and_save(path, title=None):
    if title is not None:
        plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"[FIGURE SAVED] {path}")
    plt.show()
    plt.close()

def safe_open_rgb(path: str) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except Exception as e:
        print(f"[WARN] Failed to open image: {path} | {e}")
        return Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(0, 0, 0))

def tile_image_get_quadrant(img: Image.Image, quadrant: int) -> Image.Image:
    w, h = img.size
    if quadrant == 0:
        return img.crop((0, 0, w // 2, h // 2))
    elif quadrant == 1:
        return img.crop((w // 2, 0, w, h // 2))
    elif quadrant == 2:
        return img.crop((0, h // 2, w // 2, h))
    else:
        return img.crop((w // 2, h // 2, w, h))


# ==============================================================================
# ============================ PIPELINE TRANSFORMS =============================
# ==============================================================================

tf_eval = v2.Compose([
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


# ==============================================================================
# =============================== CUSTOM ARTIFACTS =============================
# ==============================================================================

class CustomCNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 512, 3, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
    def forward(self, x):
        return self.features(x).flatten(1)

class LinearHead(nn.Module):
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)
    def forward(self, x):
        return self.fc(x)


def load_jepa_weights_into_backbone(backbone, ckpt_path):
    """Loads pre-trained LeJEPA weights cleanly into backbone without noise perturbation."""
    try:
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    except Exception:
        sd = torch.load(ckpt_path, map_location="cpu")
        
    model_dict = backbone.state_dict()
    filtered_sd = {}
    
    clean_sd = {}
    for k, v in sd.items():
        clean_key = k.replace("backbone.", "").replace("encoder.", "").replace("features.", "")
        clean_sd[clean_key] = v

    for k, v in clean_sd.items():
        if k in model_dict and v.shape == model_dict[k].shape:
            filtered_sd[k] = v

    if len(filtered_sd) == 0:
        model_keys = list(model_dict.keys())
        ckpt_keys = list(clean_sd.keys())
        matched_idx = 0
        for ck in ckpt_keys:
            if matched_idx >= len(model_keys):
                break
            mk = model_keys[matched_idx]
            if clean_sd[ck].shape == model_dict[mk].shape:
                filtered_sd[mk] = clean_sd[ck]
                matched_idx += 1

    if len(filtered_sd) > 0:
        model_dict.update(filtered_sd)
        backbone.load_state_dict(model_dict)
        print(f"✅ SUCCESS — Dynamically synchronized {len(filtered_sd)} weight maps into model backbone layers (unperturbed).")
    else:
        print("❌ WARNING — Layer match mismatch. Starting run with standard random initializations.")
        
    return backbone


# ==============================================================================
# ============================ DATA DISCOVERY AND PREPROCESSING =================
# ==============================================================================

def save_publication_grids():
    class_samples = {}
    for p in sorted(Path(CLASSIFICATION_DATASET_ROOT).iterdir()):
        if not p.is_dir() or p.name.lower().startswith(("output", "result", "log", ".", "_")):
            continue
        for fp in sorted(p.rglob("*")):
            if fp.is_file() and fp.suffix.lower() in VALID_EXTS:
                class_samples[p.name] = fp
                break

    num_classes = len(class_samples)
    cols = min(3, num_classes)
    rows = math.ceil(num_classes / cols)

    plt.rcParams.update({'font.family': 'sans-serif', 'axes.edgecolor': '#CCCCCC', 'axes.linewidth': 0.8})

    fig1, axes1 = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes1 = axes1.flatten() if num_classes > 1 else [axes1]
    for idx, (class_name, img_path) in enumerate(sorted(class_samples.items())):
        img = Image.open(img_path).convert("RGB").resize(GRID_IMAGE_SIZE, Image.Resampling.LANCZOS)
        axes1[idx].imshow(img)
        axes1[idx].set_title(class_name, fontsize=12, fontweight='bold', pad=8)
        axes1[idx].axis("off")
    for j in range(idx + 1, len(axes1)): axes1[j].axis("off")
    fig1.tight_layout()
    fig1.savefig(os.path.join(OUTPUT_ROOT, "publication_grid_original.png"), dpi=300, bbox_inches="tight")
    plt.close()

    fig2, axes2 = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes2 = axes2.flatten() if num_classes > 1 else [axes2]
    for idx, (class_name, img_path) in enumerate(sorted(class_samples.items())):
        img = Image.open(img_path).convert("RGB")
        if any(target in class_name.lower() for target in ZOOM_CLASSES):
            w, h = img.size
            left, top = w * (0.5 - CHOSEN_ZOOM / 2), h * (0.5 - CHOSEN_ZOOM / 2)
            right, bottom = w * (0.5 + CHOSEN_ZOOM / 2), h * (0.5 + CHOSEN_ZOOM / 2)
            img = img.crop((left, top, right, bottom))
        img_resized = img.resize(GRID_IMAGE_SIZE, Image.Resampling.LANCZOS)
        axes2[idx].imshow(img_resized)
        axes2[idx].set_title(class_name, fontsize=12, fontweight='bold', pad=8)
        axes2[idx].axis("off")
    for j in range(idx + 1, len(axes2)): axes2[j].axis("off")
    fig2.tight_layout()
    fig2.savefig(os.path.join(OUTPUT_ROOT, "publication_grid_processed.png"), dpi=300, bbox_inches="tight")
    plt.close()

def save_partition_preview_grids(train_df, val_df, out_path):
    """Generates 4-image preview sheets separately for training and validation subsets."""
    for label, dataframe in [("training", train_df), ("testing", val_df)]:
        sampled = dataframe.sample(n=min(4, len(dataframe)), random_state=BASE_SEED).reset_index(drop=True)
        fig, axes = plt.subplots(1, 4, figsize=(14, 4))
        for idx in range(4):
            if idx < len(sampled):
                r = sampled.iloc[idx]
                img = Image.open(r.filepath).convert("RGB")
                if any(target in r.class_name.lower() for target in ZOOM_CLASSES):
                    w, h = img.size
                    left, top = w * (0.5 - CHOSEN_ZOOM / 2), h * (0.5 - CHOSEN_ZOOM / 2)
                    right, bottom = w * (0.5 + CHOSEN_ZOOM / 2), h * (0.5 + CHOSEN_ZOOM / 2)
                    img = img.crop((left, top, right, bottom))
                axes[idx].imshow(img.resize(GRID_IMAGE_SIZE, Image.Resampling.LANCZOS))
                axes[idx].set_title(f"{r.class_name}\n({r.sample_id})", fontsize=10)
            axes[idx].axis("off")
        fig.tight_layout()
        fig.savefig(os.path.join(out_path, f"partition_preview_{label}_4_images.png"), dpi=300, bbox_inches="tight")
        plt.close()

def scan_classification_dataset(root_dir: str):
    root = Path(root_dir)
    if not root.exists(): raise FileNotFoundError(f"Root not found: {root}")
    class_dirs = []
    for p in sorted(root.iterdir()):
        if p.is_dir() and not p.name.lower().startswith(("output", "result", "log", ".", "_")):
            if any(fp.is_file() and fp.suffix.lower() in VALID_EXTS for fp in p.rglob("*")):
                class_dirs.append(p)

    class_names = [p.name for p in class_dirs]
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    rows = []
    for class_dir in class_dirs:
        for fp in sorted(class_dir.rglob("*")):
            if fp.is_file() and fp.suffix.lower() in VALID_EXTS:
                rows.append({
                    "filepath": str(fp), "class_name": class_dir.name,
                    "class_idx": class_to_idx[class_dir.name], "sample_id": fp.stem
                })
    return pd.DataFrame(rows), class_names, class_to_idx

def quadruple_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Structurally expands the dataframe row-by-row to account for all 4 quadrants explicitly."""
    expanded_rows = []
    for _, row in df.iterrows():
        for q in range(4):
            new_row = row.copy()
            new_row["quadrant_idx"] = q
            new_row["sample_id"] = f"{row['sample_id']}_quad_{q}"
            expanded_rows.append(new_row)
    return pd.DataFrame(expanded_rows).reset_index(drop=True)


# ==============================================================================
# ======================== TARGET PREPROCESSING ENVELOPE =======================
# ==============================================================================

class ZoomPreprocessingDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.tf = tf_eval

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = safe_open_rgb(r.filepath)
        
        # 1. Base Zooming Strategy
        if any(target in r.class_name.lower() for target in ZOOM_CLASSES):
            w, h = img.size
            left, top = w * (0.5 - CHOSEN_ZOOM / 2), h * (0.5 - CHOSEN_ZOOM / 2)
            right, bottom = w * (0.5 + CHOSEN_ZOOM / 2), h * (0.5 + CHOSEN_ZOOM / 2)
            img = img.crop((left, top, right, bottom))

        # 2. Quadrupled Tiling Strategy Extraction
        quadrant = int(r.get("quadrant_idx", 0))
        tile = tile_image_get_quadrant(img, quadrant)
        
        return self.tf(tile), torch.tensor(int(r.class_idx), dtype=torch.long)


# ==============================================================================
# ======================== FAST EMBEDDING CACHING ROUTINE ======================
# ==============================================================================

def extract_embeddings_and_labels(backbone, df):
    """Extracts features using frozen backbone once and caches as PyTorch tensors."""
    dataset = ZoomPreprocessingDataset(df)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    backbone.eval()
    backbone.to(DEVICE)
    
    embed_list, label_list = [], []
    with torch.no_grad():
        for x, y in loader:
            x = move_to_device(x)
            embeds = backbone(x)
            embed_list.append(embeds.cpu())
            label_list.append(y)
            
    all_embeds = torch.cat(embed_list, dim=0)
    all_labels = torch.cat(label_list, dim=0)
    return all_embeds, all_labels


def select_few_shot_indices(df: pd.DataFrame, shots: int, seed: int):
    """Selects indices of dataset based on unique original files."""
    selected_indices = []
    unique_filepaths_df = df.drop_duplicates(subset=["filepath"])
    
    for _, g in unique_filepaths_df.groupby("class_idx"):
        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n_take = min(shots, len(g))
        chosen_paths = g.iloc[:n_take]["filepath"].tolist()
        
        matching_indices = df[df["filepath"].isin(chosen_paths)].index.tolist()
        selected_indices.extend(matching_indices)
        
    random.seed(seed)
    random.shuffle(selected_indices)
    return selected_indices


# ==============================================================================
# ============================== ANALYSIS ENGINE ===============================
# ==============================================================================

def plot_pca_2d(embeddings, labels, class_names, out_path, title):
    """Generates 2D PCA scatter plot matching ResNet/MobileNet formatting."""
    pca = PCA(n_components=2, random_state=BASE_SEED)
    Z = pca.fit_transform(embeddings.numpy() if torch.is_tensor(embeddings) else embeddings)
    labels = labels.numpy() if torch.is_tensor(labels) else labels
    
    cmap = plt.get_cmap("tab20", len(class_names))
    
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in sorted(np.unique(labels)):
        mask = labels == c
        ax.scatter(Z[mask, 0], Z[mask, 1], label=class_names[c], color=cmap(c), s=22, alpha=0.85)
    
    ax.set_xlabel("PC1", fontsize=10)
    ax.set_ylabel("PC2", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(fontsize=8, loc="best", title="Classes")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[FIGURE SAVED] {out_path}")


def plot_pca_3d(embeddings, labels, class_names, out_path, title):
    """Generates 3D PCA scatter plot matching ResNet/MobileNet formatting."""
    pca = PCA(n_components=3, random_state=BASE_SEED)
    Z = pca.fit_transform(embeddings.numpy() if torch.is_tensor(embeddings) else embeddings)
    labels = labels.numpy() if torch.is_tensor(labels) else labels
    
    cmap = plt.get_cmap("tab20", len(class_names))
    
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    for c in sorted(np.unique(labels)):
        mask = labels == c
        ax.scatter(Z[mask, 0], Z[mask, 1], Z[mask, 2], label=class_names[c], color=cmap(c), s=22, alpha=0.85)
        
    ax.set_xlabel('PC1', fontsize=10, labelpad=10)
    ax.set_ylabel('PC2', fontsize=10, labelpad=10)
    ax.set_zlabel('PC3', fontsize=10, labelpad=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc='best', title="Classes")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[FIGURE SAVED] {out_path}")


def evaluate_linear_head(head, val_loader, loss_fn):
    head.eval()
    all_preds, all_trues, running_loss = [], [], 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = move_to_device(x), y.to(DEVICE)
            logits = head(x)
            running_loss += loss_fn(logits, y).item() * y.size(0)
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_trues.append(y.cpu().numpy())
            
    y_true = np.concatenate(all_trues)
    y_pred = np.concatenate(all_preds)
    return {
        "loss": running_loss / len(val_loader.dataset),
        "acc": accuracy_score(y_true, y_pred),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "y_true": y_true, "y_pred": y_pred
    }


def train_linear_head(train_embeds, train_labels, val_embeds, val_labels, in_features, num_classes, seed):
    seed_all(seed)
    train_ds = TensorDataset(train_embeds, train_labels)
    val_ds = TensorDataset(val_embeds, val_labels)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    head = LinearHead(in_features=in_features, num_classes=num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, foreach=False)
    loss_fn = nn.CrossEntropyLoss()

    best_acc, best_state = -np.inf, None

    for epoch in range(EPOCHS):
        head.train()
        for x, y in train_loader:
            x, y = move_to_device(x), y.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(head(x), y)
            loss.backward()
            optimizer.step()
        
        eval_out = evaluate_linear_head(head, val_loader, loss_fn)
        if eval_out["acc"] > best_acc:
            best_acc = eval_out["acc"]
            best_state = copy.deepcopy(head.state_dict())

    head.load_state_dict(best_state)
    return evaluate_linear_head(head, val_loader, loss_fn)


def run_classification_pipeline():
    df, class_names, class_to_idx = scan_classification_dataset(CLASSIFICATION_DATASET_ROOT)
    num_classes = len(class_names)
    
    train_df, val_df = train_test_split(df, test_size=VAL_FRAC, random_state=BASE_SEED, stratify=df["class_idx"])

    train_df_quad = quadruple_dataframe(train_df)
    val_df_quad = quadruple_dataframe(val_df)

    all_summary_rows = []

    for init_mode in INIT_MODES_TO_RUN:
        print("\n" + "="*85)
        print(f"   STARTING EVALUATION RUN FOR INITIALIZATION MODE: {init_mode.upper()}")
        print("="*85)

        classification_out_root = os.path.join(OUTPUT_ROOT, f"classification_{init_mode}")
        ensure_dir(classification_out_root)
        save_partition_preview_grids(train_df, val_df, classification_out_root)

        # ------------------------------------------------------------------
        # 1. INITIALIZE BACKBONE AND PRE-CACHE EMBEDDINGS
        # ------------------------------------------------------------------
        seed_all(BASE_SEED)
        backbone = CustomCNNBackbone()

        if init_mode == "jepa":
            backbone = load_jepa_weights_into_backbone(backbone, JEPA_CKPT)
        else:
            print(f"🎲 RANDOM BASELINE — Using pure PyTorch default initialization for backbone.")

        for param in backbone.parameters():
            param.requires_grad = False

        print(f"\n[{init_mode.upper()}] Pre-caching embeddings to accelerate linear probing...")
        train_embeds, train_labels = extract_embeddings_and_labels(backbone, train_df_quad)
        val_embeds, val_labels = extract_embeddings_and_labels(backbone, val_df_quad)
        in_features = train_embeds.shape[1]
        print(f"✅ Pre-cached embeddings: Train Shape = {train_embeds.shape}, Val Shape = {val_embeds.shape}")

        for regime_name, shots in CLASSIFICATION_REGIMES.items():
            print(f"\n[{init_mode.upper()} | REGIME] Processing Strategy Variant: {regime_name.upper()}")
            
            if shots is not None:
                seed_accs = []
                for current_seed in MULTIPLE_SEEDS:
                    shot_indices = select_few_shot_indices(train_df_quad, shots=shots, seed=current_seed)
                    sub_train_embeds = train_embeds[shot_indices]
                    sub_train_labels = train_labels[shot_indices]
                    
                    res = train_linear_head(
                        sub_train_embeds, sub_train_labels,
                        val_embeds, val_labels,
                        in_features, num_classes, current_seed
                    )
                    seed_accs.append(res["acc"])
                
                mean_val, std_val = np.mean(seed_accs), np.std(seed_accs)
                all_summary_rows.append({
                    "task": "classification", "regime": regime_name, "shots": shots, "init_mode": init_mode,
                    "accuracy_display": f"{mean_val*100:.2f}% ± {std_val*100:.2f}%", "final_val_acc": mean_val
                })
            else:
                res = train_linear_head(
                    train_embeds, train_labels,
                    val_embeds, val_labels,
                    in_features, num_classes, BASE_SEED
                )
                
                all_summary_rows.append({
                    "task": "classification", "regime": regime_name, "shots": "full", "init_mode": init_mode,
                    "accuracy_display": f"{res['acc']*100:.2f}%", "final_val_acc": res["acc"]
                })

                print(f"\n--- CLASSIFICATION REPORT FOR {init_mode.upper()} ({regime_name.upper()} REGIME) ---")
                print(classification_report(res["y_true"], res["y_pred"], target_names=class_names))

                cm = confusion_matrix(res["y_true"], res["y_pred"])
                cm_norm = confusion_matrix(res["y_true"], res["y_pred"], normalize="true")
                plt.figure(figsize=(7, 6))
                plt.imshow(cm_norm, cmap="Blues")
                plt.colorbar(label="Normalized Accuracy")
                plt.xticks(np.arange(num_classes), class_names, rotation=45, ha="right")
                plt.yticks(np.arange(num_classes), class_names)
                for i in range(num_classes):
                    for j in range(num_classes):
                        plt.text(j, i, f"{cm[i,j]}\n{cm_norm[i,j]*100:.1f}%", ha="center", va="center", color="white" if cm_norm[i,j] > 0.5 else "black")
                show_and_save(os.path.join(classification_out_root, "confusion_matrix_full.png"), f"Confusion Matrix ({init_mode.upper()})")

                # ------------------------------------------------------------------
                # 2. GENERATE UNIFIED 2D & 3D PCA FEATURE PROJECTION PLOTS
                # ------------------------------------------------------------------
                plot_pca_2d(
                    val_embeds, val_labels, class_names,
                    os.path.join(classification_out_root, "embedding_pca_2d_full.png"),
                    f"2D PCA Feature Projection Space ({init_mode.upper()})"
                )
                plot_pca_3d(
                    val_embeds, val_labels, class_names,
                    os.path.join(classification_out_root, "embedding_pca_3d_full.png"),
                    f"3D PCA Feature Projection Space ({init_mode.upper()})"
                )

    return pd.DataFrame(all_summary_rows)


# ==============================================================================
# =============================== EXECUTIVE ROUTINE ============================
# ==============================================================================

def main():
    ensure_dir(OUTPUT_ROOT)
    print(f"[ENVIRONMENT DETECTED] Backend Execution Device: {DEVICE_NAME}")
    
    save_publication_grids()

    classification_df = run_classification_pipeline()

    columns_reorder = ["task", "init_mode", "regime", "shots", "accuracy_display", "final_val_acc"]
    classification_df = classification_df.reindex(columns=columns_reorder)

    classification_df.to_csv(os.path.join(OUTPUT_ROOT, "classification_overall_summary.csv"), index=False)
    with pd.ExcelWriter(os.path.join(OUTPUT_ROOT, "classification_results_workbook.xlsx"), engine="openpyxl") as writer:
        classification_df.to_excel(writer, sheet_name="Linear Probe Metrics", index=False)

    print("\n" + "="*85 + "\nFINAL THESIS EXPERIMENTAL DATA SHEET (STABILIZED LINEAR PROBING ANALYSIS)\n" + "="*85)
    print(classification_df.to_string(index=False))

if __name__ == "__main__":
    main()

# ResNet18_EfficientNetB0_MobileNetV2_Evaluation

In [ ]:
# =========================================================
# THESIS-READY HIGH-PERFORMANCE LINEAR PROBE PIPELINE
# DIRECTML-SAFE & FEATURE-CACHED FOR AMD GPU + MULTI-CORE CPU
# =========================================================

import os
import copy
import math
import random
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torch_directml

from torchvision import models
from torchvision.transforms import v2

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.decomposition import PCA

# =========================================================
# SAFE UNPICKLING GLOBALS FOR DIRECTML BACKEND
# =========================================================
try:
    torch.serialization.add_safe_globals([torch._utils._rebuild_device_tensor_from_numpy])
except AttributeError:
    pass

# =========================================================
# SAFE IMAGE SETTINGS
# =========================================================
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =========================================================
# CONFIG (D: DRIVE INTEGRATION)
# =========================================================
CLASSIFICATION_ROOT = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\Feed Classification dataset_\Classification Dataset\Classification dataset copy 3"

LEJEPA_RESNET = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\Forageonlycheckpoints\lejepa_resnet18_tile_L2pred.pth"
LEJEPA_EFFICIENTNET = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\Forageonlycheckpoints\lejepa_efficientnetb0_tile_L2pred.pth"
LEJEPA_MOBILENET = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\Forageonlycheckpoints\lejepa_mobilenetv2_tile_L2pred.pth"

OUTPUT_DIR = r"D:\OneDrive - Michigan State University\Documents\Master thesis work\Figure"

DEVICE = torch_directml.device()
DEVICE_NAME = "directml"

IMAGE_SIZE = 224
BATCH_SIZE = 32         # Batch size for static feature extraction
HEAD_BATCH_SIZE = 128   # Batch size for linear head training
EPOCHS = 100

LR = 0.0049997148458219695
WEIGHT_DECAY = 1.8207273067144369e-06

BASE_SEED = 0
MULTIPLE_SEEDS = [0, 1, 2, 3, 4]  # 5 seeds for few-shot stabilization
NUM_WORKERS = 0                   # Optimal setting for DirectML pipeline

CLASSIFICATION_SHOTS = {
    "one_shot": 1,
    "ten_shot": 10,
    "full": None,
}

ZOOM_CLASSES = ["alfalfa", "haylage", "tmr"]
CHOSEN_ZOOM = 0.15
GRID_IMAGE_SIZE = (400, 400)
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

# =========================================================
# LOGGING / REPRODUCIBILITY
# =========================================================
def setup_logging():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    log_path = os.path.join(OUTPUT_DIR, "pipeline_classification.log")

    logger = logging.getLogger()
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    sh = logging.StreamHandler()
    sh.setFormatter(formatter)
    logger.addHandler(sh)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(BASE_SEED)

# =========================================================
# SAFE IMAGE LOADER & TRANSFORMS
# =========================================================
def safe_open(path: str):
    try:
        return Image.open(path).convert("RGB")
    except Exception as e:
        logging.warning(f"Failed to open image: {path} | {e}")
        return Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE))

eval_tf = v2.Compose([
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# =========================================================
# DATASETS & QUADRUPLING PIPELINE
# =========================================================
class ImageFolderDataset(Dataset):
    def __init__(self, root):
        self.samples = []
        root = Path(root)

        class_dirs = []
        for p in sorted(root.iterdir()):
            if not p.is_dir():
                continue
            name = p.name.lower()
            if name.startswith(("output", "result", "log", ".", "_")):
                continue

            has_valid_image = any(
                fp.is_file() and fp.suffix.lower() in VALID_EXTS
                for fp in p.rglob("*")
            )
            if has_valid_image:
                class_dirs.append(p)

        if len(class_dirs) == 0:
            raise RuntimeError(f"No valid class folders found in {root}")

        self.classes = [p.name for p in class_dirs]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for c in self.classes:
            class_dir = root / c
            for f in sorted(class_dir.rglob("*")):
                if f.is_file() and f.suffix.lower() in VALID_EXTS:
                    self.samples.append((str(f), self.class_to_idx[c]))

    def __len__(self):
        return len(self.samples)

class ZoomPreprocessingDataset(Dataset):
    """Applies center cropping on specified classes for single full images."""
    def __init__(self, base_dataset: ImageFolderDataset, subset_indices: list, tf, zoom_classes, zoom_factor):
        self.base_dataset = base_dataset
        self.subset_indices = subset_indices
        self.tf = tf
        self.zoom_classes = zoom_classes
        self.zoom_factor = zoom_factor
        self.classes = base_dataset.classes
        
        self.samples = [base_dataset.samples[i] for i in subset_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label_idx = self.samples[i]
        class_name = self.classes[label_idx].lower()
        img = safe_open(path)

        if any(target in class_name for target in self.zoom_classes):
            w, h = img.size
            left = w * (0.5 - self.zoom_factor / 2)
            top = h * (0.5 - self.zoom_factor / 2)
            right = w * (0.5 + self.zoom_factor / 2)
            bottom = h * (0.5 + self.zoom_factor / 2)
            img = img.crop((left, top, right, bottom))

        x = self.tf(img)
        return x, torch.tensor(int(label_idx), dtype=torch.long), path

class ZoomAndQuadrupleDataset(Dataset):
    """
    1. Zoom preprocessing on specified classes.
    2. Sub-divides each image into 4 tiles (Top-Left, Top-Right, Bottom-Left, Bottom-Right).
    3. Expands dataset size by 4x.
    """
    def __init__(self, base_dataset: ImageFolderDataset, subset_indices: list, tf, zoom_classes, zoom_factor):
        self.tf = tf
        self.classes = base_dataset.classes
        self.sub_samples = []

        for idx in subset_indices:
            path, label_idx = base_dataset.samples[idx]
            for tile_id in range(4):
                self.sub_samples.append((path, label_idx, tile_id))

        self.zoom_classes = zoom_classes
        self.zoom_factor = zoom_factor

    def __len__(self):
        return len(self.sub_samples)

    def __getitem__(self, i):
        path, label_idx, tile_id = self.sub_samples[i]
        class_name = self.classes[label_idx].lower()
        img = safe_open(path)

        if any(target in class_name for target in self.zoom_classes):
            w, h = img.size
            left = w * (0.5 - self.zoom_factor / 2)
            top = h * (0.5 - self.zoom_factor / 2)
            right = w * (0.5 + self.zoom_factor / 2)
            bottom = h * (0.5 + self.zoom_factor / 2)
            img = img.crop((left, top, right, bottom))

        w, h = img.size
        mid_w, mid_h = w / 2.0, h / 2.0
        
        if tile_id == 0:     # Top-Left
            box = (0, 0, mid_w, mid_h)
        elif tile_id == 1:   # Top-Right
            box = (mid_w, 0, w, mid_h)
        elif tile_id == 2:   # Bottom-Left
            box = (0, mid_h, mid_w, h)
        else:                # Bottom-Right
            box = (mid_w, mid_h, w, h)

        tile_img = img.crop(box)
        x = self.tf(tile_img)
        
        tile_path = f"{path}_tile{tile_id}"
        return x, torch.tensor(int(label_idx), dtype=torch.long), tile_path

# =========================================================
# BACKBONES
# =========================================================
class ResNet18Backbone(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        m = models.resnet18(weights=weights)
        self.features = nn.Sequential(
            m.conv1, m.bn1, m.relu, m.maxpool,
            m.layer1, m.layer2, m.layer3, m.layer4
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 512

    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)

class EfficientNetBackbone(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        m = models.efficientnet_b0(weights=weights)
        self.features = m.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 1280

    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)

class MobileNetBackbone(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        m = models.mobilenet_v2(weights=weights)
        self.features = m.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 1280

    def forward(self, x):
        return self.pool(self.features(x)).flatten(1)

def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except Exception:
        return torch.load(path, map_location=map_location, weights_only=False)

def load_lejepa(backbone, ckpt_path):
    sd = safe_torch_load(ckpt_path, map_location="cpu")
    remapped = {}
    for k, v in sd.items():
        if not k.startswith("backbone."):
            continue
        k2 = k.replace("backbone.", "")
        if isinstance(backbone, ResNet18Backbone):
            remapped[k2] = v
        else:
            remapped[f"features.{k2}"] = v
    backbone.load_state_dict(remapped, strict=False)
    return backbone

class LinearClassHead(nn.Module):
    def __init__(self, dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(dim, n_classes)
    def forward(self, x): return self.fc(x)

def freeze(model):
    for p in model.parameters(): p.requires_grad = False

def move_model(model): return model.to(DEVICE)
def move_batch_tensor(x): return x.to(DEVICE)

# =========================================================
# FEATURE CACHING ENGINE
# =========================================================
def extract_dataset_embeddings(backbone, dataset):
    """Extracts features for a given dataset."""
    freeze(backbone)
    backbone = move_model(backbone)
    backbone.eval()
    
    loader = DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=NUM_WORKERS, 
        pin_memory=False
    )
    
    all_z, all_labels, all_paths = [], [], []
    with torch.no_grad():
        for x, y, paths in loader:
            x = move_batch_tensor(x)
            z = backbone(x).detach().cpu()
            all_z.append(z)
            all_labels.append(y.cpu())
            all_paths.extend(paths)
            
    embeddings = torch.cat(all_z, dim=0)
    labels = torch.cat(all_labels, dim=0)
    return embeddings, labels, all_paths

# =========================================================
# DIAGNOSTIC FIGURES & PCA PLOTTING (MATCHING COLORMAPS)
# =========================================================
def save_train_test_sample_grids(full_dataset, train_indices, test_indices):
    plt.rcParams.update({'font.family': 'sans-serif', 'axes.edgecolor': '#CCCCCC', 'axes.linewidth': 0.8})
    
    rng_train = random.Random(BASE_SEED)
    train_samples_idx = rng_train.sample(train_indices, min(4, len(train_indices)))
    
    fig_tr, axes_tr = plt.subplots(2, 2, figsize=(6, 6))
    axes_tr = axes_tr.flatten()
    for rank, idx in enumerate(train_samples_idx):
        path, label_idx = full_dataset.samples[idx]
        class_name = full_dataset.classes[label_idx]
        img = safe_open(path).resize(GRID_IMAGE_SIZE, Image.Resampling.LANCZOS)
        axes_tr[rank].imshow(img)
        axes_tr[rank].set_title(f"Train | {class_name}", fontsize=10, fontweight='bold')
        axes_tr[rank].axis("off")
    fig_tr.tight_layout()
    fig_tr.savefig(os.path.join(OUTPUT_DIR, "partition_preview_train_4shot.png"), dpi=300, bbox_inches="tight")
    plt.close()

    rng_test = random.Random(BASE_SEED)
    test_samples_idx = rng_test.sample(test_indices, min(4, len(test_indices)))
    
    fig_ts, axes_ts = plt.subplots(2, 2, figsize=(6, 6))
    axes_ts = axes_ts.flatten()
    for rank, idx in enumerate(test_samples_idx):
        path, label_idx = full_dataset.samples[idx]
        class_name = full_dataset.classes[label_idx]
        img = safe_open(path).resize(GRID_IMAGE_SIZE, Image.Resampling.LANCZOS)
        axes_ts[rank].imshow(img)
        axes_ts[rank].set_title(f"Test | {class_name}", fontsize=10, fontweight='bold')
        axes_ts[rank].axis("off")
    fig_ts.tight_layout()
    fig_ts.savefig(os.path.join(OUTPUT_DIR, "partition_preview_test_4shot.png"), dpi=300, bbox_inches="tight")
    plt.close()

def plot_pca_2d(embeddings, labels, class_names, out_path, title):
    """Generates 2D PCA scatter plot using consistent categorical colormap."""
    pca = PCA(n_components=2, random_state=BASE_SEED)
    Z = pca.fit_transform(embeddings.numpy() if torch.is_tensor(embeddings) else embeddings)
    labels = labels.numpy() if torch.is_tensor(labels) else labels
    
    # Matching discrete colormap for class color alignment across models
    cmap = plt.get_cmap("tab20", len(class_names))
    
    fig, ax = plt.subplots(figsize=(8, 6))
    for c in sorted(np.unique(labels)):
        mask = labels == c
        ax.scatter(Z[mask, 0], Z[mask, 1], label=class_names[c], color=cmap(c), s=22, alpha=0.85)
    
    ax.set_xlabel("PC1", fontsize=10)
    ax.set_ylabel("PC2", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(fontsize=8, loc="best", title="Classes")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_pca_3d(embeddings, labels, class_names, out_path, title):
    """Generates 3D PCA scatter plot using consistent categorical colormap."""
    pca = PCA(n_components=3, random_state=BASE_SEED)
    Z = pca.fit_transform(embeddings.numpy() if torch.is_tensor(embeddings) else embeddings)
    labels = labels.numpy() if torch.is_tensor(labels) else labels
    
    # Matching discrete colormap for class color alignment across models
    cmap = plt.get_cmap("tab20", len(class_names))
    
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    for c in sorted(np.unique(labels)):
        mask = labels == c
        ax.scatter(Z[mask, 0], Z[mask, 1], Z[mask, 2], label=class_names[c], color=cmap(c), s=22, alpha=0.85)
    
    ax.set_xlabel("PC1", fontsize=10)
    ax.set_ylabel("PC2", fontsize=10)
    ax.set_zlabel("PC3", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc="best", title="Classes")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def plot_confusion_matrix(cm, class_names, out_path, title):
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(max(7, len(class_names) * 1.1), max(5, len(class_names) * 0.9)))
    plt.imshow(cm_norm, interpolation="nearest", cmap="Blues")
    plt.colorbar(label="Normalized Accuracy")
    plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(np.arange(len(class_names)), class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    thresh = cm_norm.max() / 2.0 if cm_norm.size else 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            txt = f"{cm[i,j]}\n{cm_norm[i,j]*100:.1f}%"
            plt.text(j, i, txt, ha="center", va="center", color="white" if cm_norm[i, j] > thresh else "black", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

# =========================================================
# EXPERIMENT SPLIT & SHOT SELECTION STRATEGIES
# =========================================================
def make_classification_splits(full_ds):
    y = np.array([label for _, label in full_ds.samples], dtype=int)
    idx = np.arange(len(full_ds))
    train_idx, test_idx = train_test_split(idx, test_size=0.20, random_state=BASE_SEED, stratify=y)
    return train_idx.tolist(), test_idx.tolist()

def select_unquadrupled_shot_indices(dataset: ZoomPreprocessingDataset, shots_per_class, seed):
    """
    Selects N single intact images per class from the unquadrupled dataset using a random seed.
    """
    label_to_indices = {}
    for i in range(len(dataset)):
        _, label, _ = dataset[i]
        label = int(label)
        label_to_indices.setdefault(label, []).append(i)
        
    selected_indices = []
    rng = random.Random(seed)
    for label in sorted(label_to_indices.keys()):
        inds = label_to_indices[label][:]
        rng.shuffle(inds)
        selected_indices.extend(inds[:min(shots_per_class, len(inds))])
        
    return selected_indices

# =========================================================
# ULTRA-FAST LINEAR PROBE TRAIN LOOP
# =========================================================
def train_linear_head_on_cached_features(head, train_z, train_y, test_z, test_y):
    head = move_model(head)
    
    train_loader = DataLoader(
        TensorDataset(train_z, train_y), 
        batch_size=HEAD_BATCH_SIZE, 
        shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(test_z, test_y), 
        batch_size=HEAD_BATCH_SIZE, 
        shuffle=False
    )
    
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, foreach=False)
    loss_fn = nn.CrossEntropyLoss()
    best_acc = -1.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        head.train()
        for z_batch, y_batch in train_loader:
            z_batch, y_batch = move_batch_tensor(z_batch), move_batch_tensor(y_batch)
            optimizer.zero_grad(set_to_none=True)
            out = head(z_batch)
            loss = loss_fn(out, y_batch)
            loss.backward()
            optimizer.step()

        head.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for z_batch, y_batch in test_loader:
                z_batch = move_batch_tensor(z_batch)
                preds = head(z_batch).cpu().argmax(1)
                y_true.extend(y_batch.numpy().tolist())
                y_pred.extend(preds.numpy().tolist())

        acc = accuracy_score(y_true, y_pred)
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(head.state_dict())

    head.load_state_dict(best_state)
    head.eval()
    
    y_true, y_pred = [], []
    with torch.no_grad():
        for z_batch, y_batch in test_loader:
            z_batch = move_batch_tensor(z_batch)
            preds = head(z_batch).cpu().argmax(1)
            y_true.extend(y_batch.numpy().tolist())
            y_pred.extend(preds.numpy().tolist())

    cm = confusion_matrix(y_true, y_pred)
    return accuracy_score(y_true, y_pred), cm

# =========================================================
# MAIN EXECUTIVE PIPELINE
# =========================================================
def main():
    setup_logging()
    set_seed(BASE_SEED)

    logging.info("Starting High-Performance Classification Probing Pipeline...")
    
    raw_folder_ds = ImageFolderDataset(CLASSIFICATION_ROOT)
    cls_train_idx, cls_test_idx = make_classification_splits(raw_folder_ds)
    
    save_train_test_sample_grids(raw_folder_ds, cls_train_idx, cls_test_idx)

    # 1. Single intact images dataset for Few-Shot training
    train_single_ds = ZoomPreprocessingDataset(raw_folder_ds, cls_train_idx, eval_tf, ZOOM_CLASSES, CHOSEN_ZOOM)
    
    # 2. Quadrupled dataset for Full training
    train_quad_ds = ZoomAndQuadrupleDataset(raw_folder_ds, cls_train_idx, eval_tf, ZOOM_CLASSES, CHOSEN_ZOOM)
    
    # 3. Quadrupled dataset for Test evaluation across all regimes
    test_quad_ds = ZoomAndQuadrupleDataset(raw_folder_ds, cls_test_idx, eval_tf, ZOOM_CLASSES, CHOSEN_ZOOM)

    models_dict = {
        "imagenet_resnet18": ResNet18Backbone(models.ResNet18_Weights.IMAGENET1K_V1),
        "imagenet_efficientnetb0": EfficientNetBackbone(models.EfficientNet_B0_Weights.IMAGENET1K_V1),
        "imagenet_mobilenetv2": MobileNetBackbone(models.MobileNet_V2_Weights.IMAGENET1K_V1),
        "random_resnet18": ResNet18Backbone(weights=None),
        "random_efficientnetb0": EfficientNetBackbone(weights=None),
        "random_mobilenetv2": MobileNetBackbone(weights=None),
        "lejepa_resnet18": load_lejepa(ResNet18Backbone(), LEJEPA_RESNET),
        "lejepa_efficientnetb0": load_lejepa(EfficientNetBackbone(), LEJEPA_EFFICIENTNET),
        "lejepa_mobilenetv2": load_lejepa(MobileNetBackbone(), LEJEPA_MOBILENET),
    }

    cls_results = []

    for model_name, backbone_template in models_dict.items():
        logging.info(f"Extracting static embeddings for: {model_name}")
        model_dir = os.path.join(OUTPUT_DIR, model_name)
        os.makedirs(model_dir, exist_ok=True)

        # Pre-extract single image embeddings for few-shot train set
        train_single_z, train_single_y, _ = extract_dataset_embeddings(backbone_template, train_single_ds)
        
        # Pre-extract quadrupled embeddings for full train set
        train_quad_z, train_quad_y, _ = extract_dataset_embeddings(backbone_template, train_quad_ds)
        
        # Pre-extract quadrupled embeddings for test set
        test_z, test_y, _ = extract_dataset_embeddings(backbone_template, test_quad_ds)

        model_perf = {"model": model_name}
        
        for regime_name, shots in CLASSIFICATION_SHOTS.items():
            cls_dir = os.path.join(model_dir, f"classification_{regime_name}")
            os.makedirs(cls_dir, exist_ok=True)

            if shots is not None:
                seed_accuracies = []
                for s in MULTIPLE_SEEDS:
                    set_seed(s)
                    # Sample single intact images using the seed
                    t_sub_idx = select_unquadrupled_shot_indices(train_single_ds, shots, s)
                    
                    sub_train_z = train_single_z[t_sub_idx]
                    sub_train_y = train_single_y[t_sub_idx]
                    
                    head = LinearClassHead(backbone_template.out_dim, len(raw_folder_ds.classes))
                    acc, _ = train_linear_head_on_cached_features(
                        head, sub_train_z, sub_train_y, test_z, test_y
                    )
                    seed_accuracies.append(acc)
                
                mean_acc, std_acc = np.mean(seed_accuracies), np.std(seed_accuracies)
                model_perf[regime_name] = f"{mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%"
            else:
                set_seed(BASE_SEED)
                
                # Full regime uses quadrupled training set
                head = LinearClassHead(backbone_template.out_dim, len(raw_folder_ds.classes))
                acc, cm = train_linear_head_on_cached_features(
                    head, train_quad_z, train_quad_y, test_z, test_y
                )
                model_perf[regime_name] = f"{acc * 100:.2f}%"
                
                # Confusion matrix, 2D PCA, and 3D PCA generated on quadrupled test set
                plot_confusion_matrix(
                    cm, 
                    raw_folder_ds.classes, 
                    os.path.join(cls_dir, "confusion_matrix.png"), 
                    f"{model_name} | Confusion Matrix (Quadrupled Test)"
                )
                plot_pca_2d(
                    test_z, 
                    test_y, 
                    raw_folder_ds.classes, 
                    os.path.join(cls_dir, "embedding_pca_class_2d.png"), 
                    f"{model_name} | 2D PCA Classes (Quadrupled Test)"
                )
                plot_pca_3d(
                    test_z, 
                    test_y, 
                    raw_folder_ds.classes, 
                    os.path.join(cls_dir, "embedding_pca_class_3d.png"), 
                    f"{model_name} | 3D PCA Classes (Quadrupled Test)"
                )
        
        cls_results.append(model_perf)

    # Save summary
    master_df = pd.DataFrame(cls_results)
    master_df.to_excel(os.path.join(OUTPUT_DIR, "classification_summary.xlsx"), index=False)
    
    print("\n" + "="*60 + "\nFINAL THESIS CLASSIFICATION METRICS SUMMARY\n" + "="*60)
    print(master_df.to_string(index=False))

if __name__ == "__main__":
    main()